# TERRA tutorial - spatial mapping of perturbation effects (test section)

Compute a **per-cell** perturbation score with `infer_token_distance` (W2 between each cell's unperturbed and perturbed token-embedding clouds) and **project it back onto the tissue** to see *where* the glomerular knockout has its effect.

> **Requirements:** a GPU, and `terra-st` installed with the perturbation extra:
> `pip install "terra-st[perturb]" gdown`.

This tutorial runs **end-to-end on a de-identified test kidney section we provide** using the public [`lotfollahi-lab/TERRA-96M`](https://huggingface.co/lotfollahi-lab/TERRA-96M) model. Expensive steps (tokenisation, perturbation) are **cached** locally — the first run computes them, later runs (and the other tutorials) reuse the cache.

## 1. Setup

In [ ]:
# !pip install -q "terra-st[perturb]" gdown   # uncomment on first run

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import torch
from datasets import load_from_disk

from terra import download_pretrained
from terra.inference import harmonize_adata, tokenize_adata
from terra.inference import perturb_dataset, infer_token_distance
import squidpy as sq

sc.settings.set_figure_params(dpi=80, frameon=False)
import logging; logging.basicConfig(level="INFO")  # show TERRA progress

## 2. Configuration

Set the model, the section data file, and a local cache directory. If you host the test-section `.h5ad` on Google Drive, put its file id in `DRIVE_FILE_ID` and it will be downloaded automatically.

In [ ]:
MODEL_REPO   = "lotfollahi-lab/TERRA-96M"   # HF model bundle
SECTION      = "K014"
NICHE_KEY    = "NEMO_updated_niche"           # obs column used to group / colour
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

DATA_H5AD    = "K014_full.h5ad"          # local path to the section AnnData
DRIVE_FILE_ID = "11C6z0RZXZhbKzI8SHSRbsA_k1i1Tqsfa"  # optional: gdown fallback

CACHE        = Path("cache"); CACHE.mkdir(exist_ok=True)
TOK_CACHE    = CACHE / f"{SECTION}_tokenized"          # shared across tutorials
print("device:", DEVICE)

## 3. Download the pretrained model (Hugging Face)

`download_pretrained` fetches the model bundle (`model_checkpoint.pt`, `model_config.yaml`, `token_dictionary.pkl`) and returns its local path.

In [ ]:
model_dir = download_pretrained(MODEL_REPO)
print("model bundle:", model_dir)

## 4. Load the section data

The provided test-section `.h5ad` holds raw counts (`X`), spatial coordinates (`obsm['spatial']`) and niche / cell-type labels for the whole test section (123,563 cells).

In [ ]:
if not os.path.exists(DATA_H5AD):
    import gdown
    print("downloading section data from Google Drive ...")
    gdown.download(id=DRIVE_FILE_ID, output=DATA_H5AD, quiet=False)

adata = sc.read_h5ad(DATA_H5AD)
if "cell_id" not in adata.obs.columns:
    adata.obs["cell_id"] = adata.obs_names.astype(str)
adata.obs["cell_id"] = adata.obs["cell_id"].astype(str)
print(adata)

## 5. Harmonise & tokenise (cached)

`harmonize_adata` maps gene symbols to the model's Ensembl vocabulary and filters low-quality cells/genes. `tokenize_adata` turns the neighbourhood into the gene-token sequences the model consumes. **This is the slow step**, so the result is cached — delete `cache/` to recompute, or drop in a pre-computed tokenised dataset to skip it.

In [ ]:
adata = harmonize_adata(adata)
print(f"after harmonise: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

if TOK_CACHE.exists():
    print("loading cached tokenised dataset ...")
    tok = load_from_disk(str(TOK_CACHE))
else:
    print("tokenising (first run; will be cached) ...")
    tok = tokenize_adata(adata, model_dir, str(CACHE / "tok_tmp"), nproc=4)
    tok.save_to_disk(str(TOK_CACHE))
tok.set_format("torch", columns=["rel_x_coord", "rel_y_coord", "cell_id",
                                 "gene_tokens", "gene_expr", "n_nonzero_tokens"])
print(f"tokenised cells: {len(tok):,}")

## 6. Perturb (cached)

Glomerular filtration-barrier knockout (NPNT/WT1/MAGI1) in cell + neighbourhood, all cells; `return_only_perturbed_cells=True` keeps the affected subset. Reuses the shared cache if present.

In [ ]:
genes = ["ENSG00000168743", "ENSG00000184937", "ENSG00000151276"]  # NPNT, WT1, MAGI1
perturb_df = pd.DataFrame({
    "perturbed_cell_id": "all",
    "perturbed_ensembl_id": np.repeat(genes, 2),
    "perturbation_target": ["cell", "neighborhood"] * len(genes),
    "perturbation_type": "knockout", "foldchange": np.nan,
})
PERT_CACHE = CACHE / f"{SECTION}_perturbed"
if PERT_CACHE.exists():
    dataset_perturbed = load_from_disk(str(PERT_CACHE))
else:
    dataset_perturbed = perturb_dataset(dataset=tok, perturb_df=perturb_df, model_folder_path=model_dir,
                                        nproc=1, return_only_perturbed_cells=True,
                                        pad_gene_tokens=True, adjust_positions=True)
    dataset_perturbed.save_to_disk(str(PERT_CACHE))
dataset_perturbed.set_format(type="torch")
print(f"perturbed cells: {len(dataset_perturbed):,}")

## 7. Build the aligned control

`infer_token_distance` compares each cell's unperturbed vs perturbed token cloud, so both datasets must hold the **same cells in the same order**. Subset the tokenised section to the perturbed cells, then order both by a common sorted cell-id list.

In [ ]:
perturbed_ids = set(dataset_perturbed.with_format(None)["cell_id"])
full_ids = tok.with_format(None)["cell_id"]
keep_idx = [i for i, cid in enumerate(full_ids) if cid in perturbed_ids]
control = tok.select(keep_idx)

control_ids   = control.with_format(None)["cell_id"]
perturbed_ids = dataset_perturbed.with_format(None)["cell_id"]
shared = set(control_ids) & set(perturbed_ids)
control_pos   = {cid: i for i, cid in enumerate(control_ids)   if cid in shared}
perturbed_pos = {cid: i for i, cid in enumerate(perturbed_ids) if cid in shared}
order = sorted(shared)
control           = control.select([control_pos[c]   for c in order])
dataset_perturbed = dataset_perturbed.select([perturbed_pos[c] for c in order])
control.set_format(type="torch"); dataset_perturbed.set_format(type="torch")
print(f"aligned cells: {len(order):,}")

## 8. Per-cell distance via `infer_token_distance`

 entropic-OT **Sinkhorn, `p=1`**, `blur=0.01`, `backend="tensorized"`. Returns per-cell `cell_score` / `spatial_cell_score` / `neighborhood_score`.


In [ ]:
result = infer_token_distance(
    dataset_original=control,
    dataset_perturbed=dataset_perturbed,
    model_folder_path=model_dir,
    emb_layer=None,
    batch_size=128,
    pin_memory=False,
    num_workers=12,
    loss="sinkhorn",
    p=1,
    blur=0.01,
    backend="tensorized",
    device=None,
    ignore_spc_tokens=True,
)
print({k: np.asarray(v).shape for k, v in result.items() if hasattr(v, '__len__') and not isinstance(v, dict)})

## 9. Map the per-cell score onto `adata`

Map `neighborhood_score` back onto the full section by `cell_id`; cells not in the perturbed set get 0.

In [ ]:
score_map = dict(zip(control.with_format(None)["cell_id"], result["neighborhood_score"]))

adata = sc.read_h5ad(DATA_H5AD)                      # full section for plotting
adata.obs["cell_id"] = adata.obs["cell_id"].astype(str) if "cell_id" in adata.obs else adata.obs_names.astype(str)
adata.obs["POC1_score"] = (
    adata.obs["cell_id"].map(score_map).fillna(0).astype(float).values
)

## 10. Project onto the tissue — normalized score

Per-cell score at each cell's spatial position on a symmetric diverging scale (`RdBu_r`), min-max normalized, centered at the median with a robust (98th-percentile) range. Saved as `GFB_<section>.svg`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Helvetica", "Arial"],
    "font.size": 10, "axes.labelsize": 10,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    "figure.dpi": 1200, "savefig.dpi": 1200,
})
coords = adata.obsm["spatial"]; x, y = coords[:, 0], coords[:, 1]

# symmetric scale centered at the median (min-max normalized scores)
scores = adata.obs["POC1_score"].values
scores_norm = (scores - scores.min()) / (scores.max() - scores.min())
center = np.median(scores_norm)
half_range = np.percentile(np.abs(scores_norm - center), 98)
vmin, vmax = center - half_range, center + half_range
cmap = "RdBu_r"

fig, ax = plt.subplots(figsize=(120/25.4, 110/25.4))
sc_plot = ax.scatter(x, y, c=scores_norm, cmap=cmap, vmin=vmin, vmax=vmax,
                     s=0.5, alpha=0.85, linewidths=0, rasterized=True)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"Test section \u2014 GFB", fontsize=10, pad=4)
cbar = fig.colorbar(sc_plot, ax=ax, shrink=0.4, aspect=15, pad=0.02, location="right")
cbar.set_label("score (norm)", fontsize=6, labelpad=3); cbar.ax.tick_params(labelsize=6)
cbar.outline.set_visible(False)
fig.savefig(f"GFB_{SECTION.lower()}.svg", format="svg", dpi=1200, bbox_inches="tight")
plt.show()

## 11. Project onto the tissue — raw score

The same projection on the **raw** (un-normalized) scores, symmetric around the median and clamped to the data range. Saved as `POC1_score_<section>.svg`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Helvetica", "Arial"],
    "font.size": 10, "axes.labelsize": 10,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    "figure.dpi": 1200, "savefig.dpi": 1200,
})
coords = adata.obsm["spatial"]; x, y = coords[:, 0], coords[:, 1]

# symmetric scale on RAW scores (no normalization), clamped to data range
scores = adata.obs["POC1_score"].values
center = np.median(scores)
half_range = np.percentile(np.abs(scores - center), 98)
vmin = max(scores.min(), center - half_range)
vmax = min(scores.max(), center + half_range)
cmap = "RdBu_r"

fig, ax = plt.subplots(figsize=(120/25.4, 110/25.4))
sc_plot = ax.scatter(x, y, c=scores, cmap=cmap, vmin=vmin, vmax=vmax,
                     s=0.5, alpha=0.85, linewidths=0, rasterized=True)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"Test section \u2014 POC1_score", fontsize=10, pad=4)
cbar = fig.colorbar(sc_plot, ax=ax, shrink=0.4, aspect=15, pad=0.02, location="right")
cbar.set_label("POC1_score", fontsize=6, labelpad=3); cbar.ax.tick_params(labelsize=6)
cbar.outline.set_visible(False)
fig.savefig(f"POC1_score_{SECTION.lower()}.svg", format="svg", dpi=1200, bbox_inches="tight")
plt.show()

---

## Questions / help

If you have any questions about this tutorial, please contact **Mohammad Vali Sanian** — [mohammad.sanian@helsinki.fi](mailto:mohammad.sanian@helsinki.fi) or [mv10@sanger.ac.uk](mailto:mv10@sanger.ac.uk).